# MAG-S3b в Google Colab

Среда выполнения: **TPU v6e-1** (44 ядра, OpenBLAS SkylakeX), как подтверждающий прогон MAG-S3. Если её нет, подойдёт
v5e с 24 ядрами; сверка окружения в ячейке 3 запишет разницу в последних знаках.

1. Ячейки 1 и 2 готовят среду: файлы со сверкой SHA-256, пакеты закреплённых версий, Google Диск.
2. Ячейка 3 повторяет калибровку C1–C5 и сравнивает её число за числом с эталонной калибровкой.
3. Ячейка 4 гонит слепую проверку конвейера на птицах 9401–9402 и удаляет выход непросмотренным.
4. Ячейка 5 – подтверждающий прогон, только после команды «запускай».

In [ ]:
# 1. Файлы MAG-S3b и их SHA-256
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
ROOT, WORK, DRIVE = "/content/mag_s3b", "/content/work", "/content/drive/MyDrive/MAG-S3b"
SHA256 = {
 "mag_s3b.py": "f88118db34507156d3204c00368ba14c88c884e45f17ada8855da1fc45cd0ee3",
 "estimate.py": "df2702366d55fe28c2443e5fa347d1d59709e38d571ae1254ee6eb1a81c691b0",
 "analyze_s3b.py": "a77198a499ea6e867705d6056248c510dec52568cc1f3a67c65139b62fb9e859",
 "test_s3b.py": "7185e3d0f0e96bbe58327cbc04fb01b2737725138f238c45da817fb1bf2af734",
 "mag_s3.py": "1df703af63a2af71a94385d33276ad39a422e9abaa040fdf4bcb618344e1f350",
 "analyze.py": "ecc6a15b77e4f4d1c07a4217439e0c97868b9a12971c9ada255f79840ef1d21f",
 "mag_s2.py": "2352aefe0fe5974732c318b365f4fb5477ffc47fb78b12748bca2515465829c4",
 "physics.json": "647bc0ccb1312991a1b772193cac39f4ced1481c591422c50659bb822a3100c5",
 "mag_s2_bird1.json": "0b603a65362e999a0c1da09ee0d6edc0e2864dcd690a2f277978d541943b2d86",
 "requirements.txt": "ba5ddab58d78c37fbd70099684586d8e7a59b92d99d810e8768608c7f721f3a2",
 "scripts/pipeline.sh": "f5bd18bc708013eddf9dff3f9a5d12de28ae4bd8be065ef4e4d1a7ffeaa1aa5c",
 "scripts/env_check.py": "ba793555ae12b27c3d3400405513c0710dfb1f7be6ede48b1749fc096aae487f",
 "scripts/mutate_estimate.py": "2767052d163a8d1a3675257bfd253474b9a9e6d16a215348cb701404cb7119c7",
 "reference_calibration/C1.json": "78db61a651598ebc30b43e5d7df7cb75222bb5d639e381611d643f5008d734e9",
 "reference_calibration/C2_C4.json": "bcf67a4503352430aaea3903c0b6cb131b4ce56278d575314a788e3f1f7c862c",
 "reference_calibration/C3.json": "9ee43c5c7dab396ea7b815ece370c14fe2a8be69726340a2f6a0bd2fd3ccfec0",
 "reference_calibration/C5.json": "39588e49d31b9acc085d6683b24212f9c4ab3c7123a9372d968d5d6533814b9b"
}
PAYLOAD = """
...
ci3Xci3Xci3Xci3Xcn3W138BhZRP+gCgBQA=
"""
os.makedirs(ROOT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as tar:
    tar.extractall(ROOT, filter="data")
bad = [name for name, h in SHA256.items()
       if hashlib.sha256(open(os.path.join(ROOT, name), "rb").read()).hexdigest() != h]
print("все SHA-256 совпадают с вшитыми" if not bad else f"РАСХОЖДЕНИЕ: {bad}")
assert not bad
mem = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2 ** 30
print(f"ядер: {os.cpu_count()}, память: {mem:.0f} ГБ, Python {sys.version.split()[0]}")


def sh(cmd, cwd=ROOT):
    env = dict(os.environ, WORKERS=str(os.cpu_count()))
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

In [ ]:
# 2. Пакеты закреплённых версий и Google Диск
rc = sh("set -o pipefail; pip install -q -r requirements.txt 2>&1 | tail -3")
assert rc == 0, "pip install failed"
sh("python3 -I -c 'import sys, numpy, scipy, radicalpy; print(sys.version.split()[0], numpy.__version__, "
   "scipy.__version__, radicalpy.__version__)'")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
print("Диск подключён:", DRIVE)

In [ ]:
# 3. Тесты, повтор калибровки C1-C5 и сверка окружения с эталоном (несколько минут на 44 ядрах)
t0 = time.time()
cal = f"{WORK}/calibration"
os.makedirs(cal, exist_ok=True)
rc = sh("python3 -I test_s3b.py --fast")
rc = rc or sh(f"python3 -I mag_s3b.py calibrate --out {cal} --workers {os.cpu_count()}")
rc = rc or sh(f"python3 -I scripts/env_check.py reference_calibration {cal} | tee {cal}/env_check.txt; "
              "exit ${PIPESTATUS[0]}")
if rc == 0:
    open(f"{cal}/CELL3_OK", "w").write(time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
sh(f"rm -rf {DRIVE}/calibration && cp -r {cal} {DRIVE}/calibration")
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; калибровка и сверка окружения: {DRIVE}/calibration")

In [ ]:
# 4. Слепая проверка конвейера на птицах 9401-9402; печатает только технические факты и удаляет выход
t0 = time.time()
if not os.path.exists(f"{WORK}/calibration/C1.json"):
    sh(f"cp -r {DRIVE}/calibration {WORK}/calibration")
assert os.path.exists(f"{WORK}/calibration/CELL3_OK"), "ячейка 3 не прошла: тесты, калибровка или сверка окружения"
rc = sh(f"set -o pipefail; bash scripts/pipeline.sh blind {WORK} {WORK}/calibration 1 2>&1 | tee {DRIVE}/blind_check.log")
print(f"\nкод выхода {rc} (3 – пауза по времени, запусти ячейку снова), {(time.time() - t0) / 60:.1f} мин; "
      f"журнал: {DRIVE}/blind_check.log")

Ячейка 5 запускается только после команды «запускай» и публикации SHA-256 в Telegram. Она считает частями по
полчаса и после каждой части кладёт архив `MAG-S3b/confirmatory_s3b.tar.gz` на Диск; после обрыва сессии достаточно
выполнить ячейки 1 и 2 и снова запустить её.

In [ ]:
# 5. Подтверждающий прогон MAG-S3b
t0 = time.time()
arch = f"{DRIVE}/confirmatory_s3b.tar.gz"
if not os.path.exists(f"{WORK}/calibration/C1.json"):
    sh(f"cp -r {DRIVE}/calibration {WORK}/calibration")
assert os.path.exists(f"{WORK}/calibration/CELL3_OK"), "ячейка 3 не прошла: тесты, калибровка или сверка окружения"
if os.path.exists(arch) and not os.path.exists(f"{WORK}/confirmatory_s3b"):
    sh(f"tar -xzf {arch} -C {WORK}")
while True:
    rc = sh(f"set -o pipefail; bash scripts/pipeline.sh confirmatory {WORK} {WORK}/calibration 0.5 2>&1 "
            f"| tee -a {WORK}/confirmatory.log")
    sh(f"tar -czf {WORK}/part.tar.gz -C {WORK} confirmatory_s3b && cp {WORK}/part.tar.gz {arch}.tmp && "
       f"mv {arch}.tmp {arch}")
    if rc != 3:
        break
sh(f"cp {WORK}/confirmatory.log {DRIVE}/; sha256sum {arch}")
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; архив: {arch}")